# Phase 3 — Baseline Models

Fits classical baselines for every cluster and evaluates on the test set. Results form the comparison table in Phase 5.

| Cluster | Baselines |
|---------|----------|
| C0 (Erratic) | Naive-7, TSB |
| C1a (Seasonal, STL ≥ 0.5) | Naive-7, iMAPA |
| C1b (Non-Seasonal, STL < 0.5) | Naive-7, iMAPA |
| C2 (Dense HiVol) | Naive-7, AutoETS |
| C3 (Sparse Long-Tail) | Naive-7, TSB, iMAPA |
| C4 (Ultra-Sparse) | SWLY (same-week-last-year, 364-day offset) |

**Data**: Unwinsorized train+val as history, unwinsorized test as evaluation target.  
**Metrics**: WMAPE, ε-MAPE (eps=1.0), NZ-MAPE (MAPE on nonzero-actual days only).  
Note: standard MAPE is undefined on sparse retail data with many zero-sales days (zero denominator),
so it is not reported. NZ-MAPE is the closest meaningful equivalent.


In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, TSB, IMAPA, AutoETS

DATA = Path('../data')
OUT  = DATA / 'baselines'
OUT.mkdir(exist_ok=True)

In [2]:
# ── Load data ────────────────────────────────────────────────────────────────
train_raw = pd.read_parquet(DATA / 'daily_train_clustered.parquet')
val_raw   = pd.read_parquet(DATA / 'daily_val_clustered.parquet')
test_raw  = pd.read_parquet(DATA / 'daily_test_clustered.parquet')

# Combine train+val as history for fitting (val period is part of the known past)
history = pd.concat([train_raw, val_raw], ignore_index=True)
test    = test_raw.copy()

print(f'History: {history.date.min().date()} → {history.date.max().date()}  ({history.date.nunique()} days)')
print(f'Test:    {test.date.min().date()} → {test.date.max().date()}  ({test.date.nunique()} days)')

History: 2009-12-01 → 2011-06-08  (447 days)
Test:    2011-06-09 → 2011-12-09  (157 days)


In [3]:
# ── Metric functions ─────────────────────────────────────────────────────────
# WMAPE    — volume-weighted; standard primary metric for intermittent demand
# eps_MAPE — mean APE with denominator clipped to max(|y|, 1.0); prior team standard
# MAPE     — standard APE with eps=1e-8; near-zero on days where model correctly
#            predicts 0, blows up when model predicts nonzero on zero-actual days.
#            Shows how well each model handles sparsity.
# NZ_MAPE  — MAPE restricted to nonzero-actual days only (pure sale-day accuracy)

def wmape(y_true, y_pred, eps=1.0):
    return np.sum(np.abs(y_true - y_pred)) / max(np.sum(np.maximum(np.abs(y_true), eps)), eps) * 100

def epsilon_mape(y_true, y_pred, eps=1.0):
    return np.mean(np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), eps)) * 100

def mape(y_true, y_pred, eps=1e-8):
    return np.mean(np.abs(y_true - y_pred) / np.maximum(np.abs(y_true), eps)) * 100

def nonzero_mape(y_true, y_pred):
    mask = y_true > 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / y_true[mask]) * 100

def score(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.maximum(np.asarray(y_pred, float), 0)
    return {
        'WMAPE':    round(wmape(y_true, y_pred), 2),
        'eps_MAPE': round(epsilon_mape(y_true, y_pred), 2),
        'MAPE':     round(mape(y_true, y_pred), 2),
        'NZ_MAPE':  round(nonzero_mape(y_true, y_pred), 2),
    }


In [4]:
# ── statsforecast helper ──────────────────────────────────────────────────────
def run_sf(history_df, test_df, models, cluster_label):
    h = test_df.date.nunique()

    sf_train = (
        history_df[['product_family_name', 'date', 'total_sales']]
        .rename(columns={'product_family_name': 'unique_id', 'date': 'ds', 'total_sales': 'y'})
        .sort_values(['unique_id', 'ds'])
    )

    sf = StatsForecast(models=models, freq='D', n_jobs=-1)
    sf.fit(sf_train)
    preds = sf.predict(h=h)

    test_dates = sorted(test_df.date.unique())
    skus = sf_train.unique_id.unique()
    date_idx = pd.DataFrame(
        [(sku, d) for sku in skus for d in test_dates],
        columns=['unique_id', 'ds']
    )
    preds = date_idx.merge(preds, on=['unique_id', 'ds'], how='left')

    actuals = test_df[['product_family_name', 'date', 'total_sales']].rename(
        columns={'product_family_name': 'unique_id', 'date': 'ds', 'total_sales': 'y'}
    )
    result = preds.merge(actuals, on=['unique_id', 'ds'], how='left')
    result = result.rename(columns={'unique_id': 'product_family_name', 'ds': 'date'})
    result['cluster'] = cluster_label
    return result


def eval_models(result_df, model_cols):
    rows = []
    for col in model_cols:
        m = score(result_df['y'].fillna(0).values, result_df[col].fillna(0).values)
        m['model'] = col
        rows.append(m)
    return pd.DataFrame(rows).set_index('model')[['WMAPE', 'eps_MAPE', 'MAPE', 'NZ_MAPE']]


## C0 — Erratic (n=515)
Baselines: **Naive-7** (SeasonalNaive, period=7) and **TSB** (α_d=0.15, α_p=0.10 — prior team's params)

In [5]:
c0_hist = history[history.cluster == 0]
c0_test = test[test.cluster == 0]

c0_result = run_sf(
    c0_hist, c0_test,
    models=[SeasonalNaive(season_length=7), TSB(alpha_d=0.15, alpha_p=0.10)],
    cluster_label=0
)
c0_result = c0_result.rename(columns={'SeasonalNaive': 'pred_naive7', 'TSB': 'pred_tsb'})

print('C0 baseline metrics:')
display(eval_models(c0_result, ['pred_naive7', 'pred_tsb']))
print(f'Prior team TS-LGBM: WMAPE=94.89%, ε-MAPE=340.67%, NonZero=155.94%')

C0 baseline metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_naive7,121.55,584.28,4.865709e+10,198.18
pred_tsb,108.02,597.87,5.158211e+10,167.62


Prior team TS-LGBM: WMAPE=94.89%, ε-MAPE=340.67%, NonZero=155.94%


## C1a — Seasonal (STL ≥ 0.5, n=317)
Baselines: **Naive-7** and **iMAPA**

In [6]:
c1a_hist = history[history.cluster == 1]
c1a_test = test[test.cluster == 1]

c1a_result = run_sf(
    c1a_hist, c1a_test,
    models=[SeasonalNaive(season_length=7), IMAPA()],
    cluster_label='1a'
)
c1a_result = c1a_result.rename(columns={'SeasonalNaive': 'pred_naive7', 'IMAPA': 'pred_imapa'})

print('C1a baseline metrics:')
display(eval_models(c1a_result, ['pred_naive7', 'pred_imapa']))
print('Prior team ZINB (all C1): WMAPE=126.09%, ε-MAPE=173.05%, NonZero=160.76%')

C1a baseline metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_naive7,115.12,329.1,2.782791e+10,187.18
pred_imapa,95.80,255.5,2.212958e+10,126.23


Prior team ZINB (all C1): WMAPE=126.09%, ε-MAPE=173.05%, NonZero=160.76%


## C1b — Non-Seasonal (STL < 0.5, n=156)
Baselines: **Naive-7** and **iMAPA**

In [7]:
c1b_hist = history[history.cluster == 5]
c1b_test = test[test.cluster == 5]

c1b_result = run_sf(
    c1b_hist, c1b_test,
    models=[SeasonalNaive(season_length=7), IMAPA()],
    cluster_label='1b'
)
c1b_result = c1b_result.rename(columns={'SeasonalNaive': 'pred_naive7', 'IMAPA': 'pred_imapa'})

print('C1b baseline metrics:')
display(eval_models(c1b_result, ['pred_naive7', 'pred_imapa']))

C1b baseline metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_naive7,109.17,342.65,2.941475e+10,140.80
pred_imapa,102.88,352.31,3.144158e+10,110.02


## C2 — Dense HiVol (n=10)
Baselines: **Naive-7** and **AutoETS** (Holt-Winters / ETS, automatically selects best variant)

In [8]:
c2_hist = history[history.cluster == 2]
c2_test = test[test.cluster == 2]

c2_result = run_sf(
    c2_hist, c2_test,
    models=[SeasonalNaive(season_length=7), AutoETS(season_length=7)],
    cluster_label=2
)
c2_result = c2_result.rename(columns={'SeasonalNaive': 'pred_naive7', 'AutoETS': 'pred_ets'})

print('C2 baseline metrics:')
display(eval_models(c2_result, ['pred_naive7', 'pred_ets']))
print('Prior team G-LGBM: WMAPE=76.06%, ε-MAPE=339.31%, NonZero=96.79%')

C2 baseline metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_naive7,102.55,916.34,7.437624e+10,178.61
pred_ets,93.93,1779.15,1.607523e+11,177.62


Prior team G-LGBM: WMAPE=76.06%, ε-MAPE=339.31%, NonZero=96.79%


## C3 — Sparse Long-Tail (n=1017)
Baselines: **Naive-7**, **TSB**, **iMAPA**

In [9]:
c3_hist = history[history.cluster == 3]
c3_test = test[test.cluster == 3]

c3_result = run_sf(
    c3_hist, c3_test,
    models=[SeasonalNaive(season_length=7), TSB(alpha_d=0.15, alpha_p=0.10), IMAPA()],
    cluster_label=3
)
c3_result = c3_result.rename(columns={
    'SeasonalNaive': 'pred_naive7',
    'TSB': 'pred_tsb',
    'IMAPA': 'pred_imapa'
})

print('C3 baseline metrics:')
display(eval_models(c3_result, ['pred_naive7', 'pred_tsb', 'pred_imapa']))
print('Prior team TS-HGB: WMAPE=150.11%, ε-MAPE=112.20%, NonZero=154.44%')

C3 baseline metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_naive7,111.64,137.69,1.241736e+10,137.50
pred_tsb,182.40,247.86,2.156610e+10,313.36
pred_imapa,110.48,141.44,1.296877e+10,117.78


Prior team TS-HGB: WMAPE=150.11%, ε-MAPE=112.20%, NonZero=154.44%


## C4 — Ultra-Sparse (n=8)
Baseline: **SWLY** (same-week-last-year, 364-day offset).  
C4 products have STL=1.0 (pure annual seasonal) — last year's same-DOW value is the best naive forecast.  
Note: this is also our Phase 4 production model for C4 (no better alternative for 8 ultra-sparse SKUs).

In [10]:
c4_hist = history[history.cluster == 4][['product_family_name', 'date', 'total_sales']]
c4_test = test[test.cluster == 4][['product_family_name', 'date', 'total_sales']]

# Build a lookup: for each test (sku, date), find history value at date - 364 days
hist_lookup = c4_hist.set_index(['product_family_name', 'date'])['total_sales']

def swly_pred(row):
    prior_date = row['date'] - pd.Timedelta(days=364)
    return hist_lookup.get((row['product_family_name'], prior_date), 0.0)

c4_test = c4_test.copy()
c4_test['pred_swly'] = c4_test.apply(swly_pred, axis=1)
c4_test = c4_test.rename(columns={'total_sales': 'y'})
c4_test['cluster'] = 4

# How many test rows had a valid prior-year match?
n_total = len(c4_test)
n_zero  = (c4_test['pred_swly'] == 0).sum()
print(f'C4 SWLY: {n_total - n_zero}/{n_total} test rows matched a prior-year date')

print('\nC4 SWLY metrics:')
display(eval_models(c4_test, ['pred_swly']))

C4 SWLY: 2/1256 test rows matched a prior-year date

C4 SWLY metrics:


,WMAPE,eps_MAPE,MAPE,NZ_MAPE
model,,,,
pred_swly,116.64,378.91,3.674403e+10,100.0


## Summary — All Baselines

In [11]:
# Load from saved parquets so this cell can run independently of the fit cells
from pathlib import Path
import numpy as np, pandas as pd

OUT = Path('../data/baselines')

def wmape(y, p, eps=1.0):
    return np.sum(np.abs(y-p)) / max(np.sum(np.maximum(np.abs(y), eps)), eps) * 100
def eps_mape(y, p, eps=1.0):
    return np.mean(np.abs(y-p) / np.maximum(np.abs(y), eps)) * 100
def mape(y, p, eps=1e-8):
    return np.mean(np.abs(y-p) / np.maximum(np.abs(y), eps)) * 100
def nz_mape(y, p):
    mask = y > 0
    return np.mean(np.abs(y[mask]-p[mask]) / y[mask]) * 100 if mask.sum() > 0 else np.nan
def zero_rate(y):
    return round((y == 0).mean() * 100, 1)

def score(df, col):
    y = df['y'].fillna(0).values
    p = np.maximum(df[col].fillna(0).values, 0)
    return {
        'zero_rate%': zero_rate(y),
        'WMAPE':      round(wmape(y, p), 2),
        'eps_MAPE':   round(eps_mape(y, p), 2),
        'MAPE':       round(mape(y, p), 2),
        'NZ_MAPE':    round(nz_mape(y, p), 2),
    }

files = {
    'C0':  ('c0_baselines.parquet',  ['pred_naive7', 'pred_tsb']),
    'C1a': ('c1a_baselines.parquet', ['pred_naive7', 'pred_imapa']),
    'C1b': ('c1b_baselines.parquet', ['pred_naive7', 'pred_imapa']),
    'C2':  ('c2_baselines.parquet',  ['pred_naive7', 'pred_ets']),
    'C3':  ('c3_baselines.parquet',  ['pred_naive7', 'pred_tsb', 'pred_imapa']),
    'C4':  ('c4_baselines.parquet',  ['pred_swly']),
}

rows = []
for cluster, (fname, cols) in files.items():
    df = pd.read_parquet(OUT / fname)
    best_model = None
    best_wmape = float('inf')
    for col in cols:
        m = score(df, col)
        if m['WMAPE'] < best_wmape:
            best_wmape, best_model = m['WMAPE'], col
    for col in cols:
        m = score(df, col)
        m['Cluster'] = cluster
        m['Model'] = col
        m['best'] = '★' if col == best_model else ''
        rows.append(m)

summary = pd.DataFrame(rows).set_index(['Cluster', 'Model'])
print('=== Baseline Summary (test set) ===')
print('zero_rate%: fraction of test days with zero actual sales')
print('MAPE: standard MAPE (eps=1e-8) — large when model predicts nonzero on zero-actual days')
display(summary)

print('\n=== Prior Team Production Model MAPEs (reference) ===')
prior = pd.DataFrame([
    {'Cluster': 'C0',  'Model': 'TS-LGBM', 'WMAPE': 94.89,  'eps_MAPE': 340.67, 'NZ_MAPE': 155.94},
    {'Cluster': 'C1',  'Model': 'ZINB',    'WMAPE': 126.09, 'eps_MAPE': 173.05, 'NZ_MAPE': 160.76},
    {'Cluster': 'C2',  'Model': 'G-LGBM',  'WMAPE': 76.06,  'eps_MAPE': 339.31, 'NZ_MAPE': 96.79},
    {'Cluster': 'C3',  'Model': 'TS-HGB',  'WMAPE': 150.11, 'eps_MAPE': 112.20, 'NZ_MAPE': 154.44},
]).set_index(['Cluster', 'Model'])
display(prior)


=== Baseline MAPE Summary (test set) ===
Note: MAPE uses eps=1e-8 — low when model correctly predicts 0, high when it over-predicts on zero-actual days


WMAPE  eps_MAPE          MAPE  NZ_MAPE best
Cluster Model                                                    
C0      pred_naive7  121.55    584.28  4.865709e+10   198.18     
        pred_tsb     108.02    597.87  5.158211e+10   167.62    ★
C1a     pred_naive7  115.12    329.10  2.782791e+10   187.18     
        pred_imapa    95.80    255.50  2.212958e+10   126.23    ★
C1b     pred_naive7  109.17    342.65  2.941475e+10   140.80     
        pred_imapa   102.88    352.31  3.144158e+10   110.02    ★
C2      pred_naive7  102.55    916.34  7.437624e+10   178.61     
        pred_ets      93.93   1779.15  1.607523e+11   177.62    ★
C3      pred_naive7  111.64    137.69  1.241736e+10   137.50     
        pred_tsb     182.40    247.86  2.156610e+10   313.36     
        pred_imapa   110.48    141.44  1.296877e+10   117.78    ★
C4      pred_swly    116.64    378.91  3.674403e+10   100.00    ★


=== Prior Team Production Model MAPEs (reference) ===
Note: prior team did not report standard MAPE


,,WMAPE,eps_MAPE,NZ_MAPE
Cluster,Model,,,
C0,TS-LGBM,94.89,340.67,155.94
C1,ZINB,126.09,173.05,160.76
C2,G-LGBM,76.06,339.31,96.79
C3,TS-HGB,150.11,112.20,154.44


In [12]:
# ── Save prediction parquets ──────────────────────────────────────────────────
c0_result.to_parquet(OUT / 'c0_baselines.parquet', index=False)
c1a_result.to_parquet(OUT / 'c1a_baselines.parquet', index=False)
c1b_result.to_parquet(OUT / 'c1b_baselines.parquet', index=False)
c2_result.to_parquet(OUT / 'c2_baselines.parquet', index=False)
c3_result.to_parquet(OUT / 'c3_baselines.parquet', index=False)
c4_test.to_parquet(OUT / 'c4_baselines.parquet', index=False)

print('Saved to data/baselines/')
for f in sorted(OUT.glob('*.parquet')):
    df = pd.read_parquet(f)
    print(f'  {f.name}: {df.shape}')

Saved to data/baselines/
  c0_baselines.parquet: (80855, 6)
  c1a_baselines.parquet: (49769, 6)
  c1b_baselines.parquet: (24492, 6)
  c2_baselines.parquet: (1570, 6)
  c3_baselines.parquet: (159669, 7)
  c4_baselines.parquet: (1256, 5)
